In [8]:

import torch
from torch.utils.data import DataLoader
import torch.nn as nn
import numpy as np
from utils.torch_lib.TabularTransformer import TabularTransformer, TransactionDataset


In [2]:

if torch.cuda.is_available():
    device = torch.device("cuda")
elif torch.backends.mps.is_available():
    device = torch.device("mps")
else:
    device = torch.device("cpu")

print(f"Using device: {device}")

Using device: cuda


In [3]:

CSV_TRAIN_PATH = '../../data_ieee/transactions_train.csv'
CSV_TEST_PATH = '../../data_ieee/transactions_test.csv'
PKL_PATH = '../../models/preprocessing_config.pkl'


train_dataset = TransactionDataset(CSV_TRAIN_PATH, PKL_PATH, target_col='isFraud')
test_dataset = TransactionDataset(CSV_TEST_PATH, PKL_PATH, target_col='isFraud')

loader_train = DataLoader(train_dataset, batch_size=64, shuffle=True)
loader_test = DataLoader(test_dataset, batch_size=64, shuffle=True)

for x_cat, x_cont, labels in loader_train:
    print(f"Categorical Batch Shape: {x_cat.shape}") 
    print(f"Continuous Batch Shape: {x_cont.shape}")  
    print(f"Labels Shape: {labels.shape}")            
    break

Preprocessing data... this may take a moment.
Preprocessing data... this may take a moment.
Categorical Batch Shape: torch.Size([64, 33])
Continuous Batch Shape: torch.Size([64, 170])
Labels Shape: torch.Size([64])


In [4]:
learning_rate = 1e-3
batch_size = 64
epochs = 5

In [5]:
def train_loop(dataloader, model, loss_fn, optimizer):
    size = len(dataloader.dataset)

    model.train()
    for batch, (X_cat, X_cont, y) in enumerate(dataloader):

        X_cat, X_cont, y = X_cat.to(device), X_cont.to(device), y.to(device)

        pred = model(X_cat, X_cont)
        loss = loss_fn(pred, y)


        loss.backward()
        optimizer.step()
        optimizer.zero_grad()

        if batch % 100 == 0:
            loss, current = loss.item(), batch * batch_size + len(X_cat)
            print(f"loss: {loss:>7f}  [{current:>5d}/{size:>5d}]")


def test_loop(dataloader, model, loss_fn):


    model.eval()
    size = len(dataloader.dataset)
    num_batches = len(dataloader)
    test_loss, correct = 0, 0


    with torch.no_grad():
        for (X_cat, X_cont, y) in dataloader:

            X_cat, X_cont, y = X_cat.to(device), X_cont.to(device), y.to(device)

            pred = model(X_cat, X_cont)
            test_loss += loss_fn(pred, y).item()
            correct += (pred.argmax(1) == y).type(torch.float).sum().item()

    test_loss /= num_batches
    correct /= size
    print(f"Test Error: \n Accuracy: {(100*correct):>0.1f}%, Avg loss: {test_loss:>8f} \n")

In [6]:
n_categories = train_dataset.get_n_categories()
n_continuous = train_dataset.num_idx[1]
model = TabularTransformer(n_categories=n_categories, n_continuous = n_continuous, n_classes = 2, embed_dim = 16)
model.to(device)

TabularTransformer(
  (embeddings): ModuleList(
    (0): Embedding(461397, 16)
    (1): Embedding(5, 16)
    (2): Embedding(12821, 16)
    (3-4): 2 x Embedding(5, 16)
    (5): Embedding(60, 16)
    (6): Embedding(61, 16)
    (7-9): 3 x Embedding(3, 16)
    (10): Embedding(4, 16)
    (11-16): 6 x Embedding(3, 16)
    (17): Embedding(4, 16)
    (18): Embedding(3, 16)
    (19): Embedding(4, 16)
    (20-22): 3 x Embedding(3, 16)
    (23): Embedding(76, 16)
    (24): Embedding(125, 16)
    (25): Embedding(237, 16)
    (26): Embedding(5, 16)
    (27-31): 5 x Embedding(3, 16)
    (32): Embedding(1687, 16)
  )
  (transformer): TransformerEncoder(
    (layers): ModuleList(
      (0-2): 3 x TransformerEncoderLayer(
        (self_attn): MultiheadAttention(
          (out_proj): NonDynamicallyQuantizableLinear(in_features=16, out_features=16, bias=True)
        )
        (linear1): Linear(in_features=16, out_features=2048, bias=True)
        (dropout): Dropout(p=0.1, inplace=False)
        (linear

In [9]:
loss_fn = nn.CrossEntropyLoss()
optimizer = torch.optim.SGD(model.parameters(), lr=learning_rate)

In [10]:

epochs = 2
for t in range(epochs):
    print(f"Epoch {t+1}\n-------------------------------")
    train_loop(loader_train, model, loss_fn, optimizer)
    test_loop(loader_test, model, loss_fn)
print("Done!")

Epoch 1
-------------------------------
loss: 0.848709  [   64/472432]
loss: 0.127972  [ 6464/472432]
loss: 0.138718  [12864/472432]
loss: 0.121788  [19264/472432]
loss: 0.034216  [25664/472432]
loss: 0.092244  [32064/472432]
loss: 0.112745  [38464/472432]
loss: 0.125305  [44864/472432]
loss: 0.157438  [51264/472432]
loss: 0.213018  [57664/472432]
loss: 0.101865  [64064/472432]
loss: 0.092076  [70464/472432]
loss: 0.150228  [76864/472432]
loss: 0.148279  [83264/472432]
loss: 0.148188  [89664/472432]
loss: 0.101145  [96064/472432]
loss: 0.165305  [102464/472432]
loss: 0.073544  [108864/472432]
loss: 0.161900  [115264/472432]
loss: 0.099613  [121664/472432]
loss: 0.159864  [128064/472432]
loss: 0.131997  [134464/472432]
loss: 0.150400  [140864/472432]
loss: 0.028717  [147264/472432]
loss: 0.026794  [153664/472432]
loss: 0.075361  [160064/472432]
loss: 0.037846  [166464/472432]
loss: 0.094980  [172864/472432]
loss: 0.213242  [179264/472432]
loss: 0.091918  [185664/472432]
loss: 0.105896  

In [13]:

x_cat = test_dataset.Xp_cat[0:1, :].to(device)
x_cont = test_dataset.Xp_cont[0:1, :].to(device)
y = test_dataset.y[0:1].to(device)

model.eval()
with torch.no_grad():
    pred = model(x_cat, x_cont)

print(f"Predicción para el primer elemento: {pred} | Actual : {y}")

Predicción para el primer elemento: tensor([[ 2.0235, -2.5523]], device='cuda:0') | Actual : tensor([0], device='cuda:0')


In [14]:
torch.save(model.state_dict(), '../../models/TabularTransformer_weights.pth')

In [16]:
model = TabularTransformer(n_categories=n_categories, n_continuous = n_continuous, n_classes = 2, embed_dim = 16)
model.load_state_dict(torch.load('../../models/TabularTransformer_weights.pth', weights_only=True))

<All keys matched successfully>

In [18]:
x_cat = test_dataset.Xp_cat[0:1, :].to(device)
x_cont = test_dataset.Xp_cont[0:1, :].to(device)
y = test_dataset.y[0:1].to(device)
model.to(device)
model.eval()
with torch.no_grad():
    pred = model(x_cat, x_cont)

print(f"Predicción para el primer elemento: {pred} | Actual : {y}")

Predicción para el primer elemento: tensor([[ 2.0235, -2.5523]], device='cuda:0') | Actual : tensor([0], device='cuda:0')
